In [18]:
import sys
from pathlib import Path
import requests

try:
    # .py 脚本
    current_dir = Path(__file__).resolve().parent
except NameError:
    # .ipynb
    current_dir = Path.cwd()

project_root = current_dir.parent
sys.path.append(str(project_root))

import logging
from config.log_config import setup_logging

setup_logging()
logger = logging.getLogger(__name__)

In [19]:
import os

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


def check_token_and_key():
    if not pushover_user:
        raise ValueError("enviroment variable - PUSHOVER_USER is error")
    else:
        logger.info(f"pushover_user:{pushover_user[:6]}")

    if not pushover_token:
        raise ValueError("enviroment variable - PUSHOVER_TOKEN is error")
    else:
        logger.info(f"pushover_token:{pushover_token[:6]}")

In [20]:
check_token_and_key()

2026-06-13 17:43:12,237 [INFO] __main__: pushover_user:uepm9c
2026-06-13 17:43:12,239 [INFO] __main__: pushover_token:aq9tmo


In [ ]:
def push(message: str):
    logger.info(f"push {message[:10]} to device")

    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, payload)

In [22]:
def record_user_details(
    email: str, name: str = "name not provided", notes: str = "notes not provided"
):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}


def record_unknown_question(question: str):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "如果用户有兴趣保持联系且提供了邮件地址, 那么使用此工具记录该用户",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "该用户的邮件地址",
            },
            "name": {
                "type": "string",
                "description": "该用户的姓名(如果用户确实提供了他的姓名)",
            },
            "notes": {
                "type": "string",
                "description": "记录关于本次对话的任何其他有价值的信息，以便提供背景参考",
            },
        },
        "required": ["email"],
        "additionalProperties": False,
    },
}

这里解释这个 json 协议的含义, 所有的这些都是给 ai 看的

```jsonc
record_user_details_json = {
    "name": "record_user_details", // 函数名字
    "description": "当用户表示有合作兴趣并提供了邮箱时，使用此工具记录该用户信息", // 函数说明
    "parameters": { // 函数参数
        "type": "object", // "object" 对应的类型就是 dict, 这里表示 parameters 是一个 dict 类型
        "properties": { // 函数有哪些参数, 参数类型
            "email": {
                "type": "string",
                "description": "",
            },
            "name": {"type": "string", "description": ""},
            "notes": {"type": "string", "description": ""},
        },
        "required": ["email"], // email 参数是必须传递的, 没有这个参数无法调用函数
        // 不需要额外的给其它任何参数
        // 代码中用的False, 是由于代码中的 record_user_details_json 本质是个 python-json 对象, 因而 False
        // 这里是 纯json, 因而我用了 flase
        "additionalProperties": false,
    }
}
```

In [24]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "如果一个问题你不知道如何回答或你无法回答, 使用该工具记录该问题",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "无法回答的问题"},
            "required": ["question"],
            "additionalProperties": False,
        },
    },
}

In [25]:
tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json},
]

In [26]:
from openai.types.chat import (
    ParsedChatCompletion,
    ChatCompletionMessageToolCallUnion,
    ChatCompletionMessageFunctionToolCall,
    ChatCompletionMessage,
)
import json

In [27]:
# v1
def handle_to_calls(tool_calls: list[ChatCompletionMessageToolCallUnion]):
    results: list[dict] = []
    result = "did not call any tool"
    for tool_call in tool_calls:
        if (
            isinstance(tool_call, ChatCompletionMessageFunctionToolCall)
            and tool_call.function is not None
        ):
            function = tool_call.function
            name = function.name

            logger.debug(f"function name: {name}")

            try:
                arguments = json.loads(function.arguments)

                if arguments is not None and isinstance(arguments, dict):
                    # 如果 tools 中有很多tool, 这里就要写很多条件判断
                    if name == "record_user_details":
                        result = record_user_details(**arguments)
                    elif name == "record_unknown_question":
                        result = record_unknown_question(**arguments)
                    else:
                        pass
            except TypeError as e:
                logger.error(f"模型为函数 {name} 生成了错误的参数: {e}")
                result = f"Error: Invalid arguments passed to {name}. Detail: {e}. Please correct the parameters and call again."

            results.append(
                {
                    "role": "tool",
                    "content": json.dumps(result),
                    "tool_call_id": tool_call.id,
                }
            )
    return results

```python
# tool_calls 的类型是 List[ChatCompletionMessageToolCallUnion]
# ChatCompletionMessageToolCallUnion 是 Union[ChatCompletionMessageFunctionToolCall, ChatCompletionMessageCustomToolCall]
# ChatCompletionMessageFunctionToolCall 
class ChatCompletionMessageFunctionToolCall(BaseModel):
    """A call to a function tool created by the model."""

    id: str
    """The ID of the tool call."""

    function: Function
    """The function that the model called."""

    type: Literal["function"]
    """The type of the tool. Currently, only `function` is supported."""

# Function 包含 name 和 arguments
class Function(BaseModel):
    """The function that the model called."""

    arguments: str
    """
    The arguments to call the function with, as generated by the model in JSON
    format. Note that the model does not always generate valid JSON, and may
    hallucinate parameters not defined by your function schema. Validate the
    arguments in your code before calling your function.
    """

    name: str
    """The name of the function to call."""
```

In [28]:
from pypdf import PdfReader

reader = PdfReader("./me/resume.pdf")

resume = ""

for page in reader.pages:
    if page:
        resume += page.extract_text()

logger.debug(resume[:20])

with open("./me/summary.txt", "r") as f:
    summary = f.read()

name = "魏旭阳"

2026-06-13 17:43:12,875 [DEBUG] __main__:  
 
姓       名: 魏旭阳 年


In [29]:
system_prompt = f"你正在扮演 {name}。你正在 {name} 的网站上回答问题，\
特别是与 {name} 的职业、背景、技能和经验相关的问题。\
你的职责是尽可能真实地代表 {name} 在网站上进行互动。\
系统会为你提供一份 {name} 的背景摘要及个人简历，你可以用它们来回答问题。\
请保持专业且引人入胜的语气，就像在与偶然访问网站的潜在客户或未来的雇主交谈一样。\
如果你不知道任何问题的答案，请使用你的 record_unknown_question 工具记录下这个你无法回答的问题，即使它非常琐碎或与职业无关。\
如果用户正在参与讨论，请试着引导他们通过电子邮件取得联系；询问他们的电子邮箱，并使用你的 record_user_details 工具将其记录下来。"

system_prompt += f"\n\n## 背景摘要: \n{summary}"
system_prompt += f"\n\n## 个人简历: \n{resume}"
system_prompt += f"\n\n结合上述背景，请与用户进行对话，并始终保持{name}的角色设定。"

from openai import OpenAI

qwen_client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

In [ ]:
def chat(message: str, history):
    history_clean = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = (
        [{"role": "system", "content": system_prompt}]
        + history_clean
        + [{"role": "user", "content": message}]
    )

    need_tool_call = True
    while need_tool_call:
        response = qwen_client.chat.completions.create(
            model="qwen3.7-plus",
            messages=messages,  # type:ignore
            tools=tools,  # type:ignore
        )

        response_choice = response.choices[0]

        if response_choice.finish_reason == "tool_calls":
            response_message = response_choice.message  # dict
            logger.debug(response_message)
            messages.append(response_message.model_dump(exclude_none=True))

            tool_calls = response_message.tool_calls
            if tool_calls:
                result = handle_to_calls(tool_calls)  # list[dict]
                messages.extend(result)
        else:
            need_tool_call = False

    # 仅返回了最后一次对话的 content
    return response_choice.message.content  # type:ignore

In [ ]:
import gradio as gr

gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


2026-06-13 17:44:07,677 [DEBUG] __main__: ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_f9a723d16e104fd9823dc7d6', function=Function(arguments='{"question": "你是否已婚, 是否有孩子"}', name='record_unknown_question'), type='function', index=0)], reasoning_content='用户问了一个关于个人生活的问题：“你是否已婚, 是否有孩子”。\n作为魏旭阳，我的背景摘要中并没有提到婚姻状况或是否有孩子。背景摘要主要强调了学习、工作、技术栈（Bentley, C++, OpenGL, UE, AI）以及下班后的生活（学习、刷手机、过去玩游戏现在不玩了、吃方面没有特别喜欢的）。\n由于我不知道这个问题的答案（背景信息中未提供），根据系统提示：“如果你不知道任何问题的答案，请使用你的 record_unknown_question 工具记录下这个你无法回答的问题，即使它非常琐碎或与职业无关。”\n因此，我需要：\n1. 使用 `record_unknown_question` 工具记录这个问题。\n2. 委婉地回答用户，表示这属于个人隐私，或者说明在职业网站上更倾向于讨论技术、项目和工作相关的话题，同时引导他们如果有合作或工作意向可以通过邮件联系。\n\n思考回答策略：\n- 记录问题。\n- 回答：幽默或委婉地表示这是个人隐私，或者说明网站上主要展示职业和技术背景。\n- 引导：如果有业务合作、求职或技术交流的需求，欢迎通过邮件（wxyback@foxmail.com）联系。\n\n等等，系统提示说：“如果用户正在参与讨论，请试着引导他们通过电子邮件取得联系；询问他们的电子邮箱，并使用你的 record_user_details 工具将其记录下来。”\n所以我可以先回答这个问题（委婉拒绝或说明